In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_DIM_MEDICATION_HEALTH_BEHAVIOR_INFO_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_MEDICATION_HEALTH_BEHAVIOR AS
    SELECT DISTINCT
        m.PRESCRIBED_FREQUENCY_CODE AS PRESCRIBED_FREQUENCY_CODE_AV_ID,
        av1.VALUE_SHORT_DESC AS PRESCRIBED_FREQUENCY_CODE_DESC,
        m.ADMINISTER_METHOD_CODE AS ADMINISTER_METHOD_CODE_AV_ID,
        av2.VALUE_SHORT_DESC AS ADMINISTER_METHOD_CODE_DESC,
        m.CONSENT_DECISION_CODE AS CONSENT_DECISION_CODE_AV_ID,
        av3.VALUE_SHORT_DESC AS CONSENT_DECISION_CODE_DESC,
        m.DELETED_DATE,
        SHA2(CONCAT_WS('|',
            COALESCE(TO_VARCHAR(m.PRESCRIBED_FREQUENCY_CODE), ''),
            COALESCE(av1.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(m.ADMINISTER_METHOD_CODE), ''),
            COALESCE(av2.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(m.CONSENT_DECISION_CODE), ''),
            COALESCE(av3.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(m.DELETED_DATE), '')
        )) AS CHECKSUM
    FROM {STG}.{STG_MEDICATION_HEALTH_BEHAVIOR} m
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av1
        ON m.PRESCRIBED_FREQUENCY_CODE = av1.AV_ID
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av2
        ON m.ADMINISTER_METHOD_CODE = av2.AV_ID
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av3
        ON m.CONSENT_DECISION_CODE = av3.AV_ID
    """).collect()

    session.sql(f"""
    UPDATE {DWH}.{DIM_MEDICATION_HEALTH_BEHAVIOR_INFO} tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_MEDICATION_HEALTH_BEHAVIOR src
    WHERE tgt.PRESCRIBED_FREQUENCY_CODE_AV_ID IS NOT DISTINCT FROM src.PRESCRIBED_FREQUENCY_CODE_AV_ID
      AND tgt.ADMINISTER_METHOD_CODE_AV_ID IS NOT DISTINCT FROM src.ADMINISTER_METHOD_CODE_AV_ID
      AND tgt.CONSENT_DECISION_CODE_AV_ID IS NOT DISTINCT FROM src.CONSENT_DECISION_CODE_AV_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    insert_result = session.sql(f"""
    INSERT INTO {DWH}.{DIM_MEDICATION_HEALTH_BEHAVIOR_INFO}
    (
        PRESCRIBED_FREQUENCY_CODE_AV_ID,
        PRESCRIBED_FREQUENCY_CODE_DESC,
        ADMINISTER_METHOD_CODE_AV_ID,
        ADMINISTER_METHOD_CODE_DESC,
        CONSENT_DECISION_CODE_AV_ID,
        CONSENT_DECISION_CODE_DESC,
        CREATED_DATE,
        UPDATED_DATE,
        DELETED_DATE,
        CHECKSUM
    )
    SELECT
        src.PRESCRIBED_FREQUENCY_CODE_AV_ID,
        src.PRESCRIBED_FREQUENCY_CODE_DESC,
        src.ADMINISTER_METHOD_CODE_AV_ID,
        src.ADMINISTER_METHOD_CODE_DESC,
        src.CONSENT_DECISION_CODE_AV_ID,
        src.CONSENT_DECISION_CODE_DESC,
        CURRENT_TIMESTAMP(),
        NULL,
        src.DELETED_DATE,
        src.CHECKSUM
    FROM TEMP_MEDICATION_HEALTH_BEHAVIOR src
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM {DWH}.{DIM_MEDICATION_HEALTH_BEHAVIOR_INFO} tgt
        WHERE tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_MEDICATION_HEALTH_BEHAVIOR_INFO load completed. Rows Loaded: {rows_loaded}"
    )

    print(f"DWH LOAD SUCCESS | Rows Loaded: {rows_loaded}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_MEDICATION_HEALTH_BEHAVIOR_INFO load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")